# Categorical Drift Detection Experiment

This notebook tests statistical drift detection methods for **categorical features** in the telecom churn dataset.

Two methods are implemented:

**1. Population Stability Index (PSI)**  
Measures the magnitude of distribution change between training and test datasets.

Interpretation:
- PSI < 0.1 → No drift
- 0.1 ≤ PSI < 0.25 → Moderate drift
- PSI ≥ 0.25 → Significant drift

**2. Chi-Square Test**  
Tests whether two categorical distributions are statistically different.

Hypothesis:
- H0: Train and test distributions are the same
- H1: They are different

Interpretation:
- p-value > 0.05 → No significant drift
- p-value ≤ 0.05 → Drift detected

This experiment:
1. Loads the dataset
2. Simulates a production dataset using a train/test split
3. Computes PSI and Chi-Square statistics for categorical columns
4. Generates a drift summary table

In [16]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

In [17]:
# Load dataset
train_df = pd.read_csv("public_data/train.csv")

# Preview dataset
train_df.head()

,CustomerID,UserGender,UserAge,YoungAdultFlag,RetireeStatus,Married,Dependents,NumberofDependents,Country,State,LocationCity,AreaCode,Latitude,Longitude,Population,ReferredaFriend,NumberofReferrals,TenureinMonths,Offer,VoiceService,AvgMonthlyLongDistanceCharges,AdditionalLines,ConnectivityType,InternetType,DataUsageAvg,CyberSecuritySvc,CloudStorageSvc,HardwareInsurance,PrioritySupport,VideoSvc_A,VideoSvc_B,AudioSvc,UnlimitedData,Contract,DigitalInvoicing,TransactionMode,MonthlyCharge,TotalCharges,TotalRefunds,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,CustomerLifetimeValue,ChurnStatus,Month
0,1610a102a7854c5d,Male,71,No,Yes,Yes,Yes,1,United States,California,Pine Valley,91962,32.800671,-116.483363,1604,Yes,9,38,NaN,Yes,28.30,Yes,Yes,Fiber Optic,27,No,Yes,No,No,Yes,Yes,No,Yes,One Year,No,Bank Withdrawal,101.15,3741.85,0.0,0,1075.40,4817.25,4036,No,25-Jan
1,7b4b47a335af4741,Male,77,No,Yes,Yes,No,0,United States,California,Cabazon,92230,33.929812,-116.760580,2355,Yes,5,27,NaN,Yes,33.57,Yes,Yes,DSL,21,No,Yes,Yes,Yes,Yes,Yes,No,Yes,One Year,Yes,Bank Withdrawal,84.80,2309.55,0.0,0,906.39,3215.94,5352,No,25-Jan
2,81465cd8c020404b,Male,78,No,Yes,Yes,No,0,United States,California,Desert Center,92239,33.889605,-115.257009,964,Yes,7,33,NaN,Yes,25.15,Yes,Yes,Fiber Optic,20,Yes,No,Yes,Yes,Yes,Yes,No,Yes,One Year,Yes,Bank Withdrawal,109.90,3694.70,0.0,0,829.95,4524.65,5655,No,25-Jan
3,8fb085ca69574c4c,Male,74,No,Yes,No,No,0,United States,California,Desert Hot Springs,92241,33.832799,-116.250973,5529,No,0,33,Offer C,No,0.00,No,Yes,Cable,11,No,No,Yes,Yes,Yes,Yes,No,Yes,One Year,Yes,Bank Withdrawal,54.60,1803.70,0.0,0,0.00,1803.70,5322,No,25-Jan
4,d2ca70e00e1d419f,Male,71,No,Yes,Yes,Yes,1,United States,California,Morongo Valley,92256,34.097863,-116.594561,3499,Yes,9,32,Offer C,Yes,31.86,No,Yes,Fiber Optic,23,No,Yes,No,No,Yes,Yes,Yes,Yes,One Year,Yes,Mailed Check,93.95,2861.45,0.0,0,1019.52,3880.97,2844,No,25-Jan


## Define Categorical Columns

These are the categorical features in the telecom dataset.  
They will be tested for distribution drift between datasets.

In [18]:
categorical_cols = [
    "UserGender",
    "YoungAdultFlag",
    "RetireeStatus",
    "Married",
    "Dependents",
    "Country",
    "State",
    "LocationCity",
    "ReferredaFriend",
    "Offer",
    "VoiceService",
    "AdditionalLines",
    "ConnectivityType",
    "InternetType",
    "CyberSecuritySvc",
    "CloudStorageSvc",
    "HardwareInsurance",
    "PrioritySupport",
    "VideoSvc_A",
    "VideoSvc_B",
    "AudioSvc",
    "UnlimitedData",
    "Contract",
    "DigitalInvoicing",
    "TransactionMode"
]

## Handle Missing Values

Categorical drift metrics require consistent categories.  
Missing values are replaced with a placeholder category `"Missing"`.

In [19]:
train_df[categorical_cols] = train_df[categorical_cols].fillna("Missing")

## Simulate Production Dataset

Since we only have the training dataset available, we simulate a production dataset by splitting the data.

- 70% → training baseline
- 30% → simulated production/test data

In [20]:
train_part, test_part = train_test_split(
    train_df,
    test_size=0.3,
    random_state=42
)

## Population Stability Index (PSI)

PSI measures how much a feature distribution has shifted between datasets.

Formula:

PSI = Σ (train_pct − test_pct) × ln(train_pct / test_pct)

This implementation works for categorical features by comparing category proportions.

In [21]:
def calculate_psi(train_series, test_series):

    train_dist = train_series.value_counts(normalize=True)
    test_dist = test_series.value_counts(normalize=True)

    categories = set(train_dist.index).union(set(test_dist.index))

    psi = 0

    for cat in categories:

        train_pct = train_dist.get(cat, 0.0001)
        test_pct = test_dist.get(cat, 0.0001)

        psi += (train_pct - test_pct) * np.log(train_pct / test_pct)

    return psi

## Chi-Square Drift Test

The Chi-Square test checks whether two categorical distributions differ significantly.

It uses a contingency table of category counts from train and test datasets.

In [22]:
def chi_square_test(train_series, test_series):

    train_counts = train_series.value_counts()
    test_counts = test_series.value_counts()

    categories = set(train_counts.index).union(set(test_counts.index))

    train_freq = [train_counts.get(cat, 0) for cat in categories]
    test_freq = [test_counts.get(cat, 0) for cat in categories]

    contingency_table = [train_freq, test_freq]

    chi2, p_value, _, _ = chi2_contingency(contingency_table)

    return chi2, p_value

## Drift Severity Classification

This helper function converts PSI values into drift severity categories.

In [23]:
def classify_psi(psi):

    if psi < 0.1:
        return "No Drift"

    elif psi < 0.25:
        return "Moderate Drift"

    else:
        return "Significant Drift"

## Run Drift Detection

For each categorical feature:
- Compute PSI
- Compute Chi-Square statistic
- Evaluate drift severity
- Store results in a summary table

In [24]:
results = []

for col in categorical_cols:

    psi = calculate_psi(train_part[col], test_part[col])
    chi2, p_value = chi_square_test(train_part[col], test_part[col])

    severity = classify_psi(psi)

    drift_flag = (psi > 0.25) or (p_value < 0.05)

    results.append({
        "Feature": col,
        "PSI": round(psi, 4),
        "PSI_Severity": severity,
        "Chi2_Statistic": round(chi2, 4),
        "p_value": round(p_value, 4),
        "Drift_Detected": drift_flag
    })

## Drift Summary Table

In [25]:
drift_results = pd.DataFrame(results)

drift_results.sort_values("PSI", ascending=False)

,Feature,PSI,PSI_Severity,Chi2_Statistic,p_value,Drift_Detected
7,LocationCity,0.0763,No Drift,1087.2827,0.6659,False
8,ReferredaFriend,0.0006,No Drift,8.8792,0.0029,True
3,Married,0.0004,No Drift,5.3326,0.0209,True
15,CloudStorageSvc,0.0004,No Drift,5.7044,0.0169,True
2,RetireeStatus,0.0003,No Drift,3.7804,0.0519,False
0,UserGender,0.0002,No Drift,3.2865,0.0699,False
11,AdditionalLines,0.0002,No Drift,2.6669,0.1025,False
22,Contract,0.0002,No Drift,2.6092,0.2713,False
21,UnlimitedData,0.0002,No Drift,3.6485,0.0561,False
13,InternetType,0.0002,No Drift,3.3802,0.3366,False


## Features with Detected Drift

This table shows features where drift was detected using either PSI or Chi-Square.

In [26]:
drift_results[drift_results["Drift_Detected"] == True]

,Feature,PSI,PSI_Severity,Chi2_Statistic,p_value,Drift_Detected
3,Married,0.0004,No Drift,5.3326,0.0209,True
8,ReferredaFriend,0.0006,No Drift,8.8792,0.0029,True
15,CloudStorageSvc,0.0004,No Drift,5.7044,0.0169,True


## Observations

Features with high PSI or low p-values indicate distribution shifts between datasets.

These features may require mitigation strategies such as:

- sample reweighting
- sliding window training
- feature transformation
- feature removal (for severe drift)

In the full pipeline, this drift detection module will be integrated into `drift_detector.py`.